In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================
# All imports consolidated here for the manual outlier annotation pipeline

from __future__ import annotations
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Sequence, Union
from dataclasses import dataclass
from itertools import cycle
import json
import datetime

# Core data science
import numpy as np
import pandas as pd

# Scientific computing
import scipy.signal as sig
from scipy import signal
from scipy.stats import kde

# Computer vision
import cv2

# Visualization
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns
import bokeh
import bokeh.plotting
from bokeh.io import output_notebook, show, output_file
from bokeh.plotting import figure
from bokeh.models import (
    ColumnDataSource, BoxAnnotation, HoverTool, Range1d, DataRange1d, 
    Quad, RangeTool, Div, Button, Select, TextInput, CustomJS,
    DataTable, TableColumn, NumberFormatter, StringFormatter
)
from bokeh.layouts import column, row, gridplot
from bokeh.palettes import Category10

# Project imports
from eye_tracking_system_tools.preprocessing.BlockSync_class import BlockSync
from eye_tracking_system_tools.preprocessing import utility_functions as uf

# Jupyter/IPython
%matplotlib inline

# Configuration
plt.style.use('default')
rcParams['pdf.fonttype'] = 42  # Ensure fonts are embedded and editable
rcParams['ps.fonttype'] = 42  # Ensure compatibility with vector outputs
output_notebook()  # Initialize Bokeh for notebook display
    """Generates an interactive Bokeh plot for the given data vector.
    Args:
        data_list (list or array): The data to be plotted.
        label_list (list of str): The labels of the data vectors
        plot_name (str, optional): The title of the plot. Defaults to 'default'.
        x_axis (str, optional): The label for the x-axis. Defaults to 'X'.
        y_axis (str, optional): The label for the y-axis. Defaults to 'Y'.
        peaks (list or array, optional): Indices of peaks to highlight on the plot. Defaults to None.
        export_path (False or str): when set to str, will output the resulting html fig
    """
    color_cycle = cycle(bokeh.palettes.Category10_10)
    fig = bokeh.plotting.figure(title=f'bokeh explorer: {plot_name}',
                                x_axis_label=x_axis_label,
                                y_axis_label=y_axis_label,
                                plot_width=1500,
                                plot_height=700)

    for i, data_vector in enumerate(data_list):

        color = next(color_cycle)

        if x_axis_list is None:
            x_axis = range(len(data_vector))
        elif len(x_axis_list) == len(data_list):
            print('x_axis manually set')
            x_axis = x_axis_list[i]
        else:
            raise Exception(
                'problem with x_axis_list input - should be either None, or a list with the same length as data_list')
        if label_list is None:
            fig.line(x_axis, data_vector, line_color=color, legend_label=f"Line {i + 1}")
        elif len(label_list) == len(data_list):
            fig.line(range(len(data_vector)), data_vector, line_color=color, legend_label=f"{label_list[i]}")
        if peaks is not None and peaks_list is True:
            fig.circle(peaks[i], data_vector[peaks[i]], size=10, color=color)

    if peaks is not None and peaks_list is False:
        fig.circle(peaks, data_vector[peaks], size=10, color='red')

    if export_path is not False:
        print(f'exporting to {export_path}')
        bokeh.io.output.output_file(filename=str(export_path / f'{plot_name}.html'), title=f'{plot_name}')
    bokeh.plotting.show(fig)



# create a multi-animal block_collection:

def create_block_collections(animals, block_lists, experiment_path, bad_blocks=None):
    """
    Create block collections and a block dictionary from multiple animals and their respective block lists.

    Parameters:
    - animals: list of str, names of the animals.
    - block_lists: list of lists of int, block numbers corresponding to each animal.
    - experiment_path: pathlib.Path, path to the experiment directory.
    - bad_blocks: list of int, blocks to exclude. Default is an empty list.

    Returns:
    - block_collection: list of BlockSync objects for all specified blocks.
    - block_dict: dictionary where keys are block numbers as strings and values are BlockSync objects.
    """


    if bad_blocks is None:
        bad_blocks = []

    block_collection = []
    block_dict = {}

    for animal, blocks in zip(animals, block_lists):
        # Generate blocks for the current animal
        current_blocks = uf.block_generator(
            block_numbers=blocks,
            experiment_path=experiment_path,
            animal=animal,
            bad_blocks=bad_blocks
        )
        # Add to collection and dictionary
        block_collection.extend(current_blocks)
        for b in current_blocks:
            block_dict[f"{animal}_block_{b.block_num}"] = b

    return block_collection, block_dict


# Manual Outlier Annotation Pipeline

This notebook provides tools for identifying, reviewing, and cleaning outlier data points in eye-tracking recordings.

## Workflow Overview

1. **Setup**: Load blocks and eye data
2. **Outlier Detection**: Generate threshold reports and tag outliers
3. **Review**: Manually review and tag events as good/bad
4. **Cleanup**: Apply annotations to create cleaned eye data
5. **Export**: Save cleaned data to CSV files

## Key Functions

- `outlier_threshold_report()`: Preview outlier counts at different thresholds
- `tag_outliers_to_events()`: Automatically tag outliers based on z-scores and limits
- `review_events_multi_with_arena_v2()`: Interactive video reviewer for manual annotation
- `apply_manual_outlier_cleanup()`: Apply annotations to create cleaned dataframes
- `export_clean_eye_data()`: Export cleaned data to CSV files

In [ ]:
# ============================================================================
# FUNCTION DEFINITIONS
# ============================================================================
# All functions consolidated here for the manual outlier annotation pipeline

# ---------------------------- Configuration Constants ----------------------------
CSV_NAME = "manual_event_annotations.csv"  # one file per block in its analysis folder
TAG_COL = "manual_outlier_detected"  # boolean (object-dtype), None if untagged
TS_COL = "annotation_timestamp"  # 'YYYY_MM_DD_HH_MM'

# ---------------------------- Helper Functions ----------------------------
def _now_stamp() -> str:
    """Generate timestamp string in YYYY_MM_DD_HH_MM format."""
    return pd.Timestamp.now().strftime("%Y_%m_%d_%H_%M")

def _coerce_tag(v):
    """Coerce value to boolean tag (True/False/None)."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, bool):
        return v
    s = str(v).strip().lower()
    if s in ("true", "t", "1", "yes", "y"):
        return True
    if s in ("false", "f", "0", "no", "n"):
        return False
    return None

def _block_for_key(block_dict: Dict[str, object], animal: str, block_num: Union[str, int]):
    """Find BlockSync object in block_dict by animal and block number."""
    for obj in block_dict.values():
        if getattr(obj, "animal_call", None) == animal and str(getattr(obj, "block_num", None)) == str(block_num):
            return obj
    # try direct keys e.g. "PV_126_block_7"
    key1 = f"{animal}_block_{int(block_num)}"
    key2 = f"{animal}_block_{int(block_num):03d}"
    if key1 in block_dict:
        return block_dict[key1]
    if key2 in block_dict:
        return block_dict[key2]
    raise KeyError(f"BlockSync not found for animal='{animal}', block='{block_num}'")

def _ann_path_for_block(bs) -> Path:
    """Get path to annotation CSV for a block."""
    p = Path(getattr(bs, "analysis_path"))
    p.mkdir(parents=True, exist_ok=True)
    return p / CSV_NAME

# ---------------------------- Block Collection Functions ----------------------------
def create_block_collections(animals, block_lists, experiment_path, bad_blocks=None):
    """
    Create block collections and a block dictionary from multiple animals and their respective block lists.

    Parameters:
    - animals: list of str, names of the animals.
    - block_lists: list of lists of int, block numbers corresponding to each animal.
    - experiment_path: pathlib.Path, path to the experiment directory.
    - bad_blocks: list of int, blocks to exclude. Default is an empty list.

    Returns:
    - block_collection: list of BlockSync objects for all specified blocks.
    - block_dict: dictionary where keys are block numbers as strings and values are BlockSync objects.
    """
    if bad_blocks is None:
        bad_blocks = []

    block_collection = []
    block_dict = {}

    for animal, blocks in zip(animals, block_lists):
        # Generate blocks for the current animal
        current_blocks = uf.block_generator(
            block_numbers=blocks,
            experiment_path=experiment_path,
            animal=animal,
            bad_blocks=bad_blocks
        )
        # Add to collection and dictionary
        block_collection.extend(current_blocks)
        for b in current_blocks:
            block_dict[f"{animal}_block_{b.block_num}"] = b

    return block_collection, block_dict

# ---------------------------- Annotation I/O Functions ----------------------------
def load_block_annotations(bs: object) -> pd.DataFrame:
    """
    Load per-block annotation CSV if exists. Returns standardized df:
    ['animal_call','block','eye','start_ms','end_ms','manual_outlier_detected','annotation_timestamp']
    """
    path = _ann_path_for_block(bs)
    cols = ['animal_call', 'block', 'eye', 'start_ms', 'end_ms', TAG_COL, TS_COL]
    if not path.exists():
        return pd.DataFrame(columns=cols)
    df = pd.read_csv(path)
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    df = df[cols].copy()
    # normalize dtypes
    df['animal_call'] = df['animal_call'].astype('string')
    df['block'] = df['block'].astype('string')
    df['eye'] = df['eye'].astype('string')
    df['start_ms'] = pd.to_numeric(df['start_ms'], errors='coerce').astype(float)
    df['end_ms'] = pd.to_numeric(df['end_ms'], errors='coerce').astype(float)
    df[TAG_COL] = df[TAG_COL].map(_coerce_tag).astype('object')
    df[TS_COL] = df[TS_COL].astype('string')
    return df

def _merge_annotations(old_df: pd.DataFrame, new_df: pd.DataFrame, overwrite: bool = True) -> pd.DataFrame:
    """
    Merge on (animal_call, block, eye, rounded start_ms, rounded end_ms).
    If overwrite=True, newer rows (from new_df) win.
    """
    if old_df is None or old_df.empty:
        return new_df.copy()

    out = pd.concat([old_df, new_df], ignore_index=True)
    # create integer keys for robust equality
    out['_s'] = np.rint(out['start_ms']).astype('Int64')
    out['_e'] = np.rint(out['end_ms']).astype('Int64')
    out['__ord__'] = out.index if overwrite else -out.index
    out = (out.sort_values(['animal_call', 'block', 'eye', '_s', '_e', '__ord__'])
           .drop_duplicates(['animal_call', 'block', 'eye', '_s', '_e'], keep='last')
           .drop(columns=['_s', '_e', '__ord__']))
    return out.reset_index(drop=True)

def _write_block_annotations(bs: object, df_block: pd.DataFrame, overwrite: bool = True) -> Path:
    """
    Write/merge df_block to that block's CSV.
    df_block must already be filtered to one (animal_call, block).
    """
    path = _ann_path_for_block(bs)
    existing = load_block_annotations(bs)
    final = _merge_annotations(existing, df_block, overwrite=overwrite)
    final.to_csv(path, index=False)
    return path

# ---------------------------- Export Function ----------------------------
def export_clean_eye_data(block, overwrite_original=False):
    """
    Export cleaned eye data to CSV files.
    
    This function exports left_eye_data_clean and right_eye_data_clean to CSV files.
    If overwrite_original=True, it also overwrites block.left_eye_data and block.right_eye_data
    with the cleaned versions.
    
    Parameters:
    -----------
    block : BlockSync
        BlockSync object with left_eye_data_clean and right_eye_data_clean attributes
    overwrite_original : bool
        If True, overwrites block.left_eye_data and block.right_eye_data with cleaned versions
        
    Returns:
    --------
    dict : Dictionary with paths to exported files
    """
    if not hasattr(block, 'left_eye_data_clean') or not hasattr(block, 'right_eye_data_clean'):
        raise AttributeError("Block must have left_eye_data_clean and right_eye_data_clean attributes. "
                           "Run apply_manual_outlier_cleanup() first.")
    
    # Export cleaned data
    left_path = block.analysis_path / 'left_eye_data_clean.csv'
    right_path = block.analysis_path / 'right_eye_data_clean.csv'
    
    block.left_eye_data_clean.to_csv(left_path, index=True)
    block.right_eye_data_clean.to_csv(right_path, index=True)
    
    print(f"Exported cleaned eye data:")
    print(f"  Left:  {left_path}")
    print(f"  Right: {right_path}")
    
    # Optionally overwrite original attributes
    if overwrite_original:
        block.left_eye_data = block.left_eye_data_clean.copy()
        block.right_eye_data = block.right_eye_data_clean.copy()
        print(f"\nOverwritten block.left_eye_data and block.right_eye_data with cleaned versions.")
    
    return {'left_path': left_path, 'right_path': right_path}

# ---------------------------- Interval Manipulation Functions ----------------------------
def _merge_intervals(intervals: List[Tuple[float, float]], tol: float = 0.0) -> List[Tuple[float, float]]:
    """Merge overlapping/adjacent intervals with tolerance in ms."""
    if not intervals:
        return []
    intervals = [(min(a,b), max(a,b)) for a,b in intervals if np.isfinite(a) and np.isfinite(b) and b > a]
    if not intervals:
        return []
    intervals.sort(key=lambda x: (x[0], x[1]))
    merged = [intervals[0]]
    for s,e in intervals[1:]:
        ps,pe = merged[-1]
        if s <= pe + tol:   # overlap or adjacency within tol
            merged[-1] = (ps, max(pe, e))
        else:
            merged.append((s,e))
    return merged

def _subtract_one(a: Tuple[float,float], b: Tuple[float,float], tol: float = 0.0) -> List[Tuple[float,float]]:
    """Return A \\ B (0–2 intervals). Intervals are (start,end)."""
    a0,a1 = a; b0,b1 = b
    if a1 < b0 - tol or b1 < a0 - tol:
        return [a]
    pieces = []
    left_end   = min(a1, max(a0, b0))
    right_start= max(a0, min(a1, b1))
    if left_end - a0 > tol:
        pieces.append((a0, left_end))
    if a1 - right_start > tol:
        pieces.append((right_start, a1))
    return pieces

def _subtract_many(a: Tuple[float,float], Bs: List[Tuple[float,float]], tol: float = 0.0) -> List[Tuple[float,float]]:
    """Return A \\ (union of Bs)."""
    if not Bs:
        return [a]
    Bs_merged = _merge_intervals(Bs, tol=tol)
    remaining = [a]
    for b in Bs_merged:
        next_remaining = []
        for seg in remaining:
            next_remaining.extend(_subtract_one(seg, b, tol=tol))
        remaining = next_remaining
        if not remaining:
            break
    return remaining

def _collect_existing_annotated_intervals(bs, tol_ms: float, treat_unset_as_annotated: bool=False) -> List[Tuple[float,float]]:
    """Load this block's CSV and return the union of intervals that already have a tag."""
    df = load_block_annotations(bs)
    if df is None or df.empty:
        return []
    def _has_tag(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return treat_unset_as_annotated
        s = str(v).strip().lower()
        return s in {"true","t","1","yes","y","false","f","0","no","n"}
    keep = df[df[TAG_COL].apply(_has_tag)]
    ints = [(float(s), float(e)) for s,e in zip(keep["start_ms"].values, keep["end_ms"].values)
            if np.isfinite(s) and np.isfinite(e) and e > s]
    return _merge_intervals(ints, tol=tol_ms)

# ---------------------------- Event Query Functions ----------------------------
def _runs_from_index(int_index: np.ndarray, min_run_len: int = 1, max_gap: int = 1) -> List[Tuple[int, int]]:
    """
    From a sorted array of integer indices, return [(start_idx, end_idx)] for contiguous runs.
    max_gap=1 means consecutive indices (diff == 1) are one run; larger max_gap stitches small breaks.
    """
    if int_index.size == 0:
        return []
    diffs = np.diff(int_index)
    boundaries = np.where(diffs > max_gap)[0]
    starts = np.r_[0, boundaries + 1]
    ends = np.r_[boundaries, len(int_index) - 1]
    runs = [(int(int_index[s]), int(int_index[e])) for s, e in zip(starts, ends)]
    if min_run_len > 1:
        runs = [r for r in runs if (r[1] - r[0] + 1) >= min_run_len]
    return runs

def _pad_ms_bounds(df: pd.DataFrame, start_ms: float, end_ms: float, pre_pad_ms: float, post_pad_ms: float) -> Tuple[float, float]:
    """Pad time bounds while respecting DataFrame bounds."""
    if "ms_axis" not in df.columns:
        return start_ms, end_ms
    ms = df["ms_axis"].values
    s = start_ms - float(pre_pad_ms)
    e = end_ms + float(post_pad_ms)
    if ms.size:
        s = max(min(ms[0], ms[-1]), min(s, max(ms[0], ms[-1])))
        e = max(min(ms[0], ms[-1]), min(e, max(ms[0], ms[-1])))
    return float(s), float(e)

def query_to_events_df(
        eye_df: pd.DataFrame,
        query_str: str,
        *,
        animal: str,
        block: Union[int, str],
        eye: str,
        pre_pad_ms: float = 0.0,
        post_pad_ms: float = 0.0,
        min_run_len: int = 1,
        max_gap: int = 1,
        sort_by_start: bool = True,
) -> pd.DataFrame:
    """
    Turn a DataFrame slice via df.query(...) into lumped contiguous events for manual review.
    
    Returns DataFrame with: ['animal','block','eye','start_ms','end_ms']
    """
    if eye not in ("L", "R", "left", "right"):
        raise ValueError("eye should be 'L'/'R' (or 'left'/'right').")
    
    eye_short = "L" if eye.lower().startswith("l") else "R"
    
    try:
        sub = eye_df.query(query_str)
    except Exception as e:
        raise ValueError(f"query failed: {e}")
    
    if sub.empty:
        return pd.DataFrame(columns=["animal", "block", "eye", "start_ms", "end_ms"])
    
    if not np.issubdtype(sub.index.dtype, np.integer):
        pos_idx = eye_df.index.get_indexer(sub.index)
        valid = pos_idx >= 0
        int_idx = pos_idx[valid]
        sub = sub.iloc[np.where(valid)[0]]
    else:
        int_idx = sub.index.values
    
    int_idx = np.asarray(int_idx, dtype=int)
    runs = _runs_from_index(np.sort(int_idx), min_run_len=min_run_len, max_gap=max_gap)
    
    rows = []
    for i0, i1 in runs:
        try:
            r0 = eye_df.loc[i0]
            r1 = eye_df.loc[i1]
        except KeyError:
            r0 = eye_df.iloc[i0] if (0 <= i0 < len(eye_df)) else sub.iloc[0]
            r1 = eye_df.iloc[i1] if (0 <= i1 < len(eye_df)) else sub.iloc[-1]
        
        if "ms_axis" in eye_df.columns:
            s_ms = float(r0["ms_axis"])
            e_ms = float(r1["ms_axis"])
        else:
            s_ms = float(i0)
            e_ms = float(i1)
        
        s_ms, e_ms = _pad_ms_bounds(eye_df, s_ms, e_ms, pre_pad_ms, post_pad_ms)
        if e_ms <= s_ms:
            continue
        
        rows.append({
            "animal": str(animal),
            "block": str(block),
            "eye": eye_short,
            "start_ms": s_ms,
            "end_ms": e_ms,
        })
    
    out = pd.DataFrame(rows)
    if sort_by_start and not out.empty:
        out = out.sort_values(["animal", "block", "eye", "start_ms"]).reset_index(drop=True)
    return out

def threshold_to_events_df(
        eye_df: pd.DataFrame,
        column: str,
        *,
        op: str,
        value: float,
        animal: str,
        block: Union[int, str],
        eye: str,
        pre_pad_ms: float = 0.0,
        post_pad_ms: float = 0.0,
        min_run_len: int = 1,
        max_gap: int = 1,
        sort_by_start: bool = True,
) -> pd.DataFrame:
    """
    Build events by thresholding a single column without writing a query string.
    op: one of '>', '>=', '<', '<=', '==', '!=', 'abs>'
    """
    if column not in eye_df.columns:
        raise ValueError(f"column '{column}' not found in DataFrame.")
    
    if op == 'abs>':
        q = f"abs({column}) > @value"
    elif op in ('>', '>=', '<', '<=', '==', '!='):
        q = f"{column} {op} @value"
    else:
        raise ValueError("op must be one of {'>','>=','<','<=','==','!=','abs>'}")
    
    return query_to_events_df(
        eye_df, q,
        animal=animal, block=block, eye=eye,
        pre_pad_ms=pre_pad_ms, post_pad_ms=post_pad_ms,
        min_run_len=min_run_len, max_gap=max_gap, sort_by_start=sort_by_start
    )

def trim_events_against_existing_annotations(
    block_dict: Dict[str, object],
    events_df: pd.DataFrame,
    *,
    animal_col: str = "animal",
    block_col: str = "block",
    eye_col: str = "eye",
    start_col: str = "start_ms",
    end_col: str = "end_ms",
    ms_tolerance: float = 1.0,
    treat_unset_as_annotated: bool = False,
) -> pd.DataFrame:
    """
    Subtract per-block, already-annotated intervals from incoming events_df.
    Splits partially overlapping events so that only never-seen timestamps remain for review.
    """
    if events_df.empty:
        return events_df.copy()
    
    E = events_df.copy()
    for c in (animal_col, block_col, eye_col):
        E[c] = E[c].astype(str)
    E[start_col] = pd.to_numeric(E[start_col], errors="coerce").astype(float)
    E[end_col] = pd.to_numeric(E[end_col], errors="coerce").astype(float)
    E = E.dropna(subset=[start_col, end_col])
    E = E[E[end_col] > E[start_col]].reset_index(drop=True)
    
    out_rows = []
    for (animal, block), idxs in E.groupby([animal_col, block_col]).indices.items():
        bs = _block_for_key(block_dict, animal, block)
        annotated_union = _collect_existing_annotated_intervals(bs, tol_ms=ms_tolerance,
                                                                treat_unset_as_annotated=treat_unset_as_annotated)
        if not annotated_union:
            out_rows.extend(E.loc[idxs].to_dict("records"))
            continue
        
        for i in idxs:
            s = float(E.at[i, start_col])
            e = float(E.at[i, end_col])
            segs = _subtract_many((s,e), annotated_union, tol=ms_tolerance)
            if not segs:
                continue
            for (ns, ne) in segs:
                rec = E.loc[i].to_dict()
                rec[start_col] = float(ns)
                rec[end_col] = float(ne)
                out_rows.append(rec)
    
    trimmed = pd.DataFrame(out_rows)
    if trimmed.empty:
        return trimmed
    trimmed = trimmed[trimmed[end_col] > trimmed[start_col] + 1e-9]
    trimmed = trimmed.sort_values([animal_col, block_col, start_col, end_col]).reset_index(drop=True)
    return trimmed

# Note: Large functions (review_events_multi_with_arena_v2, outlier detection functions,
# interactive_manual_tagger_js, apply_manual_outlier_cleanup, etc.) remain in their
# original cells below to keep the function cell manageable. They are documented where used.

## Step 1: Block Setup and Data Loading

This cell:
- Defines which animals and blocks to process
- Creates BlockSync objects for each block
- Loads preprocessed eye data from CSV files (created by previous preprocessing steps)
- Calibrates pupil diameter if needed

**Note**: This assumes you've already run the synchronization and verification pipelines. The eye data should be loaded from files like `left_eye_data_degrees_raw_verified.csv` and `right_eye_data_degrees_raw_verified.csv`.

In [ ]:
# BLOCK DEFINITION #

animals = ['PV_106' ]
block_lists = [[15]]
experiment_path = Path(r"D:\sample_data_for_eye_repo")
bad_blocks = [0]  # Example of bad blocks

block_collection, block_dict = create_block_collections(
    animals=animals,
    block_lists=block_lists,
    experiment_path=experiment_path,
    bad_blocks=bad_blocks
)
for block in block_collection:
    block.parse_open_ephys_events()
    block.get_eye_brightness_vectors()
    block.synchronize_block()
    block.create_eye_brightness_df(threshold_value=20)

    # if the code fails here, go to manual synchronization
    block.import_manual_sync_df()
    block.read_dlc_data()
    block.calibrate_pixel_size(10)
    #load_eye_data_2d_w_rotation_matrix(block) #should be integrated again... later

    for block in block_collection:
        block.left_eye_data = pd.read_csv(block.analysis_path / f'left_eye_data_degrees_raw_verified.csv')
        block.right_eye_data = pd.read_csv(block.analysis_path / 'right_eye_data_degrees_raw_verified.csv')

    # calibrate pupil diameter:
    if 'pupil_diameter' not in block.left_eye_data.columns:
        block.left_eye_data['pupil_diameter_pixels'] = block.left_eye_data.major_ax
        block.right_eye_data['pupil_diameter_pixels'] = block.right_eye_data.major_ax
        block.left_eye_data['pupil_diameter'] = block.left_eye_data['pupil_diameter_pixels'] * block.L_pix_size
        block.right_eye_data['pupil_diameter'] = block.right_eye_data['pupil_diameter_pixels'] * block.R_pix_size

In [ ]:
## Step 2: Event Trimming (Optional)

This step removes events that have already been annotated from the events dataframe, so you only review new/unseen events.

**Note**: The `trim_events_against_existing_annotations()` function is defined in the function cell (cell 2). Use this cell to trim your events before manual review.

### Example Usage:

```python
# Trim events to only show unannotated intervals
events_df_trimmed = trim_events_against_existing_annotations(
    block_dict, 
    events_df,
    ms_tolerance=1.0,                # matches your tolerant matching in the viewer
    treat_unset_as_annotated=False   # change to True if you also want to skip previously UNSET rows
)
```


In [ ]:
# ============================================================================
# Large Functions for Outlier Detection and Manual Annotation
# ============================================================================
# These large functions remain in this cell to keep the main function cell (cell 2) manageable.
# They include: review_events_multi_with_arena_v2, outlier_threshold_report, 
# tag_outliers_to_events, bokeh_verify_outliers, interactive_manual_tagger_js,
# apply_manual_outlier_cleanup, integrate_manual_events_into_block_annotations

# Note: Duplicate helper functions and constants have been removed - use functions from cell 2.

# === Outlier reporting, tagging, and Bokeh verification ===
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Optional, Sequence, Union
from dataclasses import dataclass
from itertools import chain
from bokeh.plotting import figure, show
from bokeh.layouts import gridplot, column
from bokeh.models import ColumnDataSource, BoxAnnotation, HoverTool
from bokeh.io import output_notebook

output_notebook()  # comment out if you prefer standalone HTML files only


# ------------------------------ configuration dataclass ------------------------------
@dataclass
class EyeColumns:
    ms: str = "ms_axis"
    phi: str = "k_phi"             # angular elevation (deg)
    theta: str = "k_theta"         # angular azimuth (deg)
    pupil: str = "pupil_diameter"  # units: your pipeline's pupil metric
    frame: Optional[str] = None    # optional, not required here


# ------------------------------ robust zscore ------------------------------
def _robust_zscore(x: pd.Series) -> pd.Series:
    """Median/MAD z-score (less sensitive to tails); NaN-safe."""
    x = pd.to_numeric(x, errors="coerce")
    med = np.nanmedian(x.values)
    mad = np.nanmedian(np.abs(x.values - med))
    scale = 1.4826 * (mad if mad > 0 else (np.nanstd(x.values) if np.nanstd(x.values) > 0 else 1.0))
    return (x - med) / scale


# ------------------------------ helpers: boolean runs -> (start_ms, end_ms) ------------------------------
def _boolean_runs_to_events(ms: np.ndarray, mask: np.ndarray, bridge_ms: float) -> List[Tuple[float, float]]:
    """
    Convert a boolean mask over ms-axis into merged [start_ms, end_ms] events, stitching gaps <= bridge_ms.
    """
    ms = ms.astype(float)
    mask = mask.astype(bool)
    if ms.size == 0 or not mask.any():
        return []

    # find initial contiguous runs where mask is True
    idx = np.flatnonzero(mask)
    # split on gaps > 1 in index (contiguity by sampling steps)
    split_points = np.where(np.diff(idx) > 1)[0]
    starts = np.r_[0, split_points + 1]
    ends = np.r_[split_points, len(idx) - 1]

    raw_events = [(ms[idx[s]], ms[idx[e]]) for s, e in zip(starts, ends)]
    if not raw_events:
        return []

    # stitch neighboring events if the gap between them ≤ bridge_ms
    merged = [raw_events[0]]
    for s, e in raw_events[1:]:
        prev_s, prev_e = merged[-1]
        gap = max(0.0, s - prev_e)
        if gap <= float(bridge_ms):
            merged[-1] = (prev_s, e)
        else:
            merged.append((s, e))
    return merged


def _overlap(a: Tuple[float, float], b: Tuple[float, float], tol_ms: float = 0.0) -> bool:
    return not (a[1] < b[0] - tol_ms or b[1] < a[0] - tol_ms)


def _merge_binocular(
    events_L: List[Tuple[float, float, str]],
    events_R: List[Tuple[float, float, str]],
    tol_ms: float = 1.0
) -> List[Tuple[float, float, str, str]]:
    """
    Merge L/R event lists (start,end,source) into possibly binocular LR events.
    Returns list of (start_ms, end_ms, eye_label, source_merged).
    Merging rule: if any L and R event overlap within tol_ms -> one LR event with unioned sources.
    Otherwise keep single-eye events.
    """
    out: List[Tuple[float, float, str, str]] = []
    used_R = np.zeros(len(events_R), dtype=bool)

    for sL, eL, srcL in events_L:
        merged_flag = False
        for j, (sR, eR, srcR) in enumerate(events_R):
            if used_R[j]:
                continue
            if _overlap((sL, eL), (sR, eR), tol_ms=tol_ms):
                s = float(min(sL, sR))
                e = float(max(eL, eR))
                src = f"L[{srcL}] + R[{srcR}]"
                out.append((s, e, "LR", src))
                used_R[j] = True
                merged_flag = True
                break
        if not merged_flag:
            out.append((float(sL), float(eL), "L", f"L[{srcL}]"))

    # append remaining R-only
    for (sR, eR, srcR), used in zip(events_R, used_R):
        if not used:
            out.append((float(sR), float(eR), "R", f"R[{srcR}]"))

    # stable sort by start
    out.sort(key=lambda r: (r[0], r[1]))
    return out


# ------------------------------ (1) threshold sweep report ------------------------------
def outlier_threshold_report(
    block,
    z_abs_list: Sequence[float],
    cols: EyeColumns = EyeColumns(),
    physiol_limits: Dict[str, Tuple[Optional[float], Optional[float]]] = None,
    eyes: Sequence[str] = ("L", "R")
) -> pd.DataFrame:
    """
    For each |z| in z_abs_list, report how many samples would be flagged per eye & signal.
    physiol_limits: dict with keys in {'phi','theta','pupil'} -> (min,max) hard bounds (None to disable a side).
    Returns tidy DataFrame for inspection.
    """
    physiol_limits = physiol_limits or {}
    rows = []

    eye_map = {
        "L": getattr(block, "left_eye_data"),
        "R": getattr(block, "right_eye_data"),
    }
    for eye in eyes:
        df = eye_map[eye]
        # ensure columns exist
        for c in (cols.ms, cols.phi, cols.theta, cols.pupil):
            if c not in df.columns:
                raise ValueError(f"Column '{c}' missing for eye {eye}")

        z_phi = _robust_zscore(df[cols.phi])
        z_theta = _robust_zscore(df[cols.theta])
        z_pupil = _robust_zscore(df[cols.pupil])

        # hard limits
        def lim_mask(signal: str) -> np.ndarray:
            v = df[getattr(cols, signal)]
            lo, hi = physiol_limits.get(signal, (None, None))
            lo_ok = np.full(len(v), True) if lo is None else (v >= float(lo))
            hi_ok = np.full(len(v), True) if hi is None else (v <= float(hi))
            return ~(lo_ok & hi_ok)  # True where violates limits

        hard_phi = lim_mask("phi")
        hard_theta = lim_mask("theta")
        hard_pupil = lim_mask("pupil")

        for zthr in z_abs_list:
            rel_phi = np.abs(z_phi.values) > float(zthr)
            rel_theta = np.abs(z_theta.values) > float(zthr)
            rel_pupil = np.abs(z_pupil.values) > float(zthr)
            rows.extend([
                {"animal": block.animal_call, "block": str(block.block_num), "eye": eye, "signal": "phi",
                 "criterion": f"|z|>{zthr}", "count": int(rel_phi.sum()), "percent": 100*rel_phi.mean()},
                {"animal": block.animal_call, "block": str(block.block_num), "eye": eye, "signal": "theta",
                 "criterion": f"|z|>{zthr}", "count": int(rel_theta.sum()), "percent": 100*rel_theta.mean()},
                {"animal": block.animal_call, "block": str(block.block_num), "eye": eye, "signal": "pupil",
                 "criterion": f"|z|>{zthr}", "count": int(rel_pupil.sum()), "percent": 100*rel_pupil.mean()},
            ])

        # add hard-limit rows once (criterion label 'limits')
        rows.extend([
            {"animal": block.animal_call, "block": str(block.block_num), "eye": eye, "signal": "phi",
             "criterion": "limits", "count": int(hard_phi.sum()), "percent": 100*hard_phi.mean()},
            {"animal": block.animal_call, "block": str(block.block_num), "eye": eye, "signal": "theta",
             "criterion": "limits", "count": int(hard_theta.sum()), "percent": 100*hard_theta.mean()},
            {"animal": block.animal_call, "block": str(block.block_num), "eye": eye, "signal": "pupil",
             "criterion": "limits", "count": int(hard_pupil.sum()), "percent": 100*hard_pupil.mean()},
        ])

    rep = pd.DataFrame(rows)
    return rep.sort_values(["eye", "signal", "criterion"]).reset_index(drop=True)


# ------------------------------ (2) tagging with bridging + sources ------------------------------
def tag_outliers_to_events(
    block,
    z_abs_threshold: float,
    cols: EyeColumns = EyeColumns(),
    physiol_limits: Dict[str, Tuple[Optional[float], Optional[float]]] = None,
    bridge_ms: float = 50.0,
    binocular_merge: bool = True,
    binocular_tol_ms: float = 1.0,
    min_duration_ms: float = 0.0,
    source_mode: str = "union",   # 'union' or 'max'
    *,
    phi_limits: Optional[Tuple[Optional[float], Optional[float]]] = None,
    theta_limits: Optional[Tuple[Optional[float], Optional[float]]] = None,
    pupil_limits: Optional[Tuple[Optional[float], Optional[float]]] = None,
) -> pd.DataFrame:
    """
    Creates an events dataframe:
      ['animal','block','eye','start_ms','end_ms','source']

    Flags samples by OR of:
      • (|z| > z_abs_threshold) for {phi, theta, pupil}
      • OR outside hard limits for any provided limits
          - precedence: explicit *limits arguments override entries in physiol_limits

    Then:
      • Builds events per eye from the boolean mask, stitching gaps <= bridge_ms.
      • Optionally merges L/R overlaps into a single 'LR' event (within binocular_tol_ms).
      • 'source' records which signals triggered within each event.

    Parameters added:
      phi_limits, theta_limits, pupil_limits:
         Tuples of (min, max). Use None to disable a side, e.g. (-40, None).
         If provided, they override the corresponding key in physiol_limits.
    """
    # ---- resolve effective hard limits (explicit args override dict) ----
    physiol_limits = dict(physiol_limits or {})
    if phi_limits is not None:
        physiol_limits["phi"] = phi_limits
    if theta_limits is not None:
        physiol_limits["theta"] = theta_limits
    if pupil_limits is not None:
        physiol_limits["pupil"] = pupil_limits

    def build_eye_events(df: pd.DataFrame, eye_label: str) -> List[Tuple[float, float, str]]:
        # robust z
        z_phi = _robust_zscore(df[cols.phi])
        z_theta = _robust_zscore(df[cols.theta])
        z_pupil = _robust_zscore(df[cols.pupil])

        rel_phi = np.abs(z_phi.values) > float(z_abs_threshold)
        rel_theta = np.abs(z_theta.values) > float(z_abs_threshold)
        rel_pupil = np.abs(z_pupil.values) > float(z_abs_threshold)

        ms = pd.to_numeric(df[cols.ms], errors="coerce").values.astype(float)

        # hard-limit masks (True where value violates limits)
        def hard(signal: str) -> np.ndarray:
            v = pd.to_numeric(df[getattr(cols, signal)], errors="coerce").values
            lo, hi = physiol_limits.get(signal, (None, None))
            lo_bad = np.zeros_like(v, dtype=bool) if lo is None else (v < float(lo))
            hi_bad = np.zeros_like(v, dtype=bool) if hi is None else (v > float(hi))
            return lo_bad | hi_bad

        hard_phi = hard("phi")
        hard_theta = hard("theta")
        hard_pupil = hard("pupil")

        # union mask across all signals/criteria
        mask = rel_phi | rel_theta | rel_pupil | hard_phi | hard_theta | hard_pupil

        # derive events with bridging
        intervals = _boolean_runs_to_events(ms, mask, bridge_ms=bridge_ms)

        # attribute sources inside each interval
        out: List[Tuple[float, float, str]] = []
        for s, e in intervals:
            in_evt = (ms >= s) & (ms <= e)

            causes = []
            if rel_phi[in_evt].any():    causes.append("phi_z")
            if rel_theta[in_evt].any():  causes.append("theta_z")
            if rel_pupil[in_evt].any():  causes.append("pupil_z")
            if hard_phi[in_evt].any():   causes.append("phi_limit")
            if hard_theta[in_evt].any(): causes.append("theta_limit")
            if hard_pupil[in_evt].any(): causes.append("pupil_limit")
            if not causes:
                causes = ["unknown"]

            if float(e - s) >= float(min_duration_ms):
                if source_mode == "max":
                    # pick dominant trigger by sample count
                    counts = {
                        "phi_z": int(rel_phi[in_evt].sum()),
                        "theta_z": int(rel_theta[in_evt].sum()),
                        "pupil_z": int(rel_pupil[in_evt].sum()),
                        "phi_limit": int(hard_phi[in_evt].sum()),
                        "theta_limit": int(hard_theta[in_evt].sum()),
                        "pupil_limit": int(hard_pupil[in_evt].sum()),
                    }
                    # keep only keys present in causes to avoid zero-only winners
                    counts = {k: v for k, v in counts.items() if k in causes}
                    top = max(counts, key=counts.get) if counts else "unknown"
                    src = top
                else:
                    src = "+".join(sorted(set(causes)))
                out.append((float(s), float(e), src))
        return out

    # sanity checks for required columns
    le_df = getattr(block, "left_eye_data")
    re_df = getattr(block, "right_eye_data")
    for eye_df in (le_df, re_df):
        for c in (cols.ms, cols.phi, cols.theta, cols.pupil):
            if c not in eye_df.columns:
                raise ValueError(f"Required column '{c}' missing in eye dataframe.")

    events_L = build_eye_events(le_df, "L")
    events_R = build_eye_events(re_df, "R")

    # binocular merge (overlap -> LR)
    if binocular_merge:
        merged = _merge_binocular(events_L, events_R, tol_ms=binocular_tol_ms)
    else:
        merged = ([(s, e, "L", f"L[{src}]") for (s, e, src) in events_L] +
                  [(s, e, "R", f"R[{src}]") for (s, e, src) in events_R])
        merged.sort(key=lambda r: (r[0], r[1]))

    out_df = pd.DataFrame({
        "animal": str(block.animal_call),
        "block": str(block.block_num),
        "eye": [lab for (_, _, lab, _) in merged],
        "start_ms": [s for (s, _, _, _) in merged],
        "end_ms": [e for (_, e, _, _) in merged],
        "source": [src for (_, _, _, src) in merged],
    })
    return out_df[["animal", "block", "eye", "start_ms", "end_ms", "source"]]


# ------------------------------ (3) Bokeh verification plotter ------------------------------
def bokeh_verify_outliers(
    block,
    events_df: pd.DataFrame,
    cols: EyeColumns = EyeColumns(),
    title: str = "Eye signals with tagged intervals",
    export_html: Optional[str] = None  # path to write a standalone HTML (optional)
):
    """
    Creates an interactive Bokeh view:
      - Left/Right eye: phi(t), theta(t), pupil(t) on shared x-range (ms).
      - Tagged intervals shown as shaded spans; L, R, and LR use different colors.
      - Pan/zoom linked across all plots.
    """
    # pull data
    L = getattr(block, "left_eye_data")
    R = getattr(block, "right_eye_data")
    for df in (L, R):
        for c in (cols.ms, cols.phi, cols.theta, cols.pupil):
            if c not in df.columns:
                raise ValueError(f"Column '{c}' missing for verification: {c}")

    srcL = ColumnDataSource(dict(
        ms=L[cols.ms].astype(float),
        phi=pd.to_numeric(L[cols.phi], errors="coerce"),
        theta=pd.to_numeric(L[cols.theta], errors="coerce"),
        pupil=pd.to_numeric(L[cols.pupil], errors="coerce"),
    ))
    srcR = ColumnDataSource(dict(
        ms=R[cols.ms].astype(float),
        phi=pd.to_numeric(R[cols.phi], errors="coerce"),
        theta=pd.to_numeric(R[cols.theta], errors="coerce"),
        pupil=pd.to_numeric(R[cols.pupil], errors="coerce"),
    ))

    # figures (shared x_range)
    pL_phi = figure(title=f"{title} – Left φ", width=1200, height=180, tools="pan,wheel_zoom,box_zoom,reset,save",
                    x_axis_label="time (ms)")
    pL_theta = figure(title="Left θ", width=1200, height=180, x_range=pL_phi.x_range,
                      tools="pan,wheel_zoom,box_zoom,reset,save")
    pL_pupil = figure(title="Left pupil", width=1200, height=180, x_range=pL_phi.x_range,
                      tools="pan,wheel_zoom,box_zoom,reset,save", x_axis_label="time (ms)")

    pR_phi = figure(title="Right φ", width=1200, height=180, x_range=pL_phi.x_range,
                    tools="pan,wheel_zoom,box_zoom,reset,save")
    pR_theta = figure(title="Right θ", width=1200, height=180, x_range=pL_phi.x_range,
                      tools="pan,wheel_zoom,box_zoom,reset,save")
    pR_pupil = figure(title="Right pupil", width=1200, height=180, x_range=pL_phi.x_range,
                      tools="pan,wheel_zoom,box_zoom,reset,save", x_axis_label="time (ms)")

    for p in (pL_phi, pL_theta, pL_pupil, pR_phi, pR_theta, pR_pupil):
        p.add_tools(HoverTool(tooltips=[("t (ms)", "@ms")], mode="vline"))

    # draw lines
    pL_phi.line("ms", "phi", source=srcL)
    pL_theta.line("ms", "theta", source=srcL)
    pL_pupil.line("ms", "pupil", source=srcL)

    pR_phi.line("ms", "phi", source=srcR)
    pR_theta.line("ms", "theta", source=srcR)
    pR_pupil.line("ms", "pupil", source=srcR)

    # add shaded tags
    def add_spans(figs, start_ms, end_ms, eye):
        if eye == "L":
            ba = BoxAnnotation(left=start_ms, right=end_ms, fill_alpha=0.18, fill_color="red")
            for p in figs[:3]:  # left panels
                p.add_layout(ba)
        elif eye == "R":
            ba = BoxAnnotation(left=start_ms, right=end_ms, fill_alpha=0.18, fill_color="blue")
            for p in figs[3:]:  # right panels
                p.add_layout(ba)
        else:  # LR
            ba = BoxAnnotation(left=start_ms, right=end_ms, fill_alpha=0.12, fill_color="purple")
            for p in figs:
                p.add_layout(ba)

    figs = [pL_phi, pL_theta, pL_pupil, pR_phi, pR_theta, pR_pupil]
    for _, row in events_df.iterrows():
        add_spans(figs, float(row["start_ms"]), float(row["end_ms"]), str(row["eye"]).upper())

    layout = gridplot([[pL_phi], [pL_theta], [pL_pupil], [pR_phi], [pR_theta], [pR_pupil]], toolbar_location="above")
    if export_html:
        from bokeh.io import output_file
        output_file(export_html, title=title)
    show(layout)


## Step 4: Outlier Detection Functions

This cell contains large functions for automatic outlier detection:
- `outlier_threshold_report()`: Generate a report showing outlier counts at different z-score thresholds
- `tag_outliers_to_events()`: Automatically tag outliers based on z-scores and physiological limits
- `bokeh_verify_outliers()`: Visual verification of tagged outliers using Bokeh plots
- `review_events_multi_with_arena_v2()`: Interactive video reviewer for manual annotation
- `interactive_manual_tagger_js()`: JavaScript-based interactive tagger
- `apply_manual_outlier_cleanup()`: Apply annotations to create cleaned dataframes
- `integrate_manual_events_into_block_annotations()`: Integrate manual events into per-block CSV

**Note**: These large functions remain in this cell to keep the main function cell (cell 2) manageable.

## Step 5: Outlier Detection Workflow

This section guides you through automatic outlier detection:

1. **Generate threshold report**: Preview how many outliers would be detected at different z-score thresholds
2. **Tag outliers**: Automatically create events dataframe from detected outliers
3. **Trim existing annotations**: Remove already-annotated events (optional)
4. **Review events**: Manually review and tag events as good/bad using the interactive reviewer
5. **Visual verification**: View tagged outliers in Bokeh plots (optional)

### 5.1: Generate Outlier Threshold Report

This cell generates a report showing how many outliers would be detected at different z-score thresholds. Use this to choose an appropriate threshold before tagging outliers.

In [ ]:
# === Example usage with a single BlockSync object ===

# 0) choose your block and column mapping if different names are used
cols = EyeColumns(
    ms="ms_axis",
    phi="k_phi",
    theta="k_theta",
    pupil="pupil_diameter",
)

# 1) sweep report to help pick a threshold
z_list = [2.0, 2.5, 3.0, 3.5, 4.0,4.5]  # absolute robust z-scores to preview
phys_limits = {
    # Use None to disable a side; set your physiology-based bounds here (deg / diameter unit)
    "phi":   (-30.0, 30.0),
    "theta": (-45.0, 25.0),
    "pupil": (1.5, 2.5),
}
report_df = outlier_threshold_report(block, z_abs_list=z_list, cols=cols, physiol_limits=phys_limits)
display(report_df.head(20))

### 5.2: Tag Outliers to Events

This cell automatically tags outliers based on the chosen z-score threshold and physiological limits. It creates an events dataframe with detected outlier intervals.

In [ ]:
# 2) tag events with your chosen parameters
events_df = tag_outliers_to_events(
    block,
    z_abs_threshold=4.5,     # pick based on the report
    cols=cols,
    physiol_limits=phys_limits,
    bridge_ms=50.0,          # contiguous if gaps <= 50 ms
    binocular_merge=True,    # collapse overlapping L/R into LR
    binocular_tol_ms=1.0,    # overlap tolerance when merging
    min_duration_ms=0.0,     # drop micro events if you want
    source_mode="union",phi_limits=(-40,40),theta_limits=(-40,40)     # 'union' (collect all causes) or 'max' (dominant cause)
)

### 5.3: Trim Events Against Existing Annotations (Optional)

This cell removes events that have already been annotated, so you only review new/unseen events. This is useful when re-running the detection pipeline.

In [ ]:
# events_df = tag_outliers_to_events(...)  # your current builder or any other source
events_df_trimmed = trim_events_against_existing_annotations(
    block_dict, events_df,
    ms_tolerance=1.0,                # matches your tolerant matching in the viewer
    treat_unset_as_annotated=False   # change to True if you also want to skip previously UNSET rows
)

### 5.4: Review Events with Interactive Video Reviewer

This cell launches the interactive video reviewer (`review_events_multi_with_arena_v2`) where you can:
- View left/right eye videos and arena video (if available)
- Navigate through events (Prev/Next, Play/Pause)
- Mark events as BAD (keyboard: 'b') or GOOD (keyboard: 'g')
- Export annotations to per-block CSV files

**Controls**: See function documentation in cell 7 for full control details.

In [ ]:

# Then review only *new* time slices:
reviewed_df = review_events_multi_with_arena_v2(block_dict, events_df_trimmed)


### 5.5: Visual Verification with Bokeh (Optional)

This cell creates an interactive Bokeh plot showing the eye tracking signals with tagged outlier intervals highlighted. Useful for verifying that the automatic detection worked correctly.

In [ ]:
# 3) visual verification (opens interactive Bokeh; shaded spans show L/R/LR tags)

bokeh_verify_outliers(block, events_df_trimmed, cols=cols, title=f"{block.animal_call} B{block.block_num} verification",export_html=block.analysis_path / 'outlier_tags_verifier.html')

## Step 6: Interactive Manual Tagger (Alternative Method)

This section provides an alternative method for manual annotation using a JavaScript-based interactive tagger.

### 6.1: Launch Interactive Manual Tagger

This cell launches `interactive_manual_tagger_js()`, a JavaScript-based interactive tool for manually selecting and tagging time intervals. You can:
- Navigate through the data using the range tool
- Add/delete events
- Export events as CSV or copy as JSON
- Tag events with eye labels (L/R/LR) and notes

In [ ]:
# 4) (optional) launch your existing manual reviewer for final GOOD/BAD triage
# from your provided functions (must be in the same kernel/session):
reviewed_df = review_events_multi_with_arena_v2(block_dict, events_df)
# reviewed_df will include tri-state 'manual_outlier_detected' + timestamps and write per-block CSV on export.


### 6.2: Paste Manual Events from JSON

After using the interactive tagger, copy the JSON output and paste it here. This creates an events dataframe that can be reviewed with the video reviewer.

In [ ]:
# --- Interactive manual tagger (JS-only, with DataTable + CSV/JSON export) ---

import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Optional

from bokeh.io import show, output_notebook
from bokeh.plotting import figure
from bokeh.layouts import column, gridplot, row
from bokeh.models import (
    ColumnDataSource, Range1d, DataRange1d, Quad, RangeTool,
    HoverTool, Div, Button, Select, TextInput, CustomJS,
    DataTable, TableColumn, NumberFormatter, StringFormatter
)

output_notebook()

@dataclass
class EyeColumns:
    ms: str = "ms_axis"
    phi: str = "k_phi"
    theta: str = "k_theta"
    pupil: str = "pupil_diameter"

def interactive_manual_tagger_js(
    block,
    cols: EyeColumns = EyeColumns(),
    preload_df: Optional[pd.DataFrame] = None,
    default_eye: str = "LR",
    width: int = 1200,
    lane_height: int = 250,
    nav_height: int = 140,
):
    # ---------- helpers ----------
    def _finite_arr(a):
        a = pd.to_numeric(pd.Series(a), errors="coerce").astype(float).values
        m = np.isfinite(a)
        return a[m] if m.any() else np.array([0.0], dtype=float)

    def _finite_xy(x, y):
        x = pd.to_numeric(pd.Series(x), errors="coerce").astype(float).values
        y = pd.to_numeric(pd.Series(y), errors="coerce").astype(float).values
        n = min(len(x), len(y))
        if n == 0:
            return np.array([0.0], dtype=float), np.array([0.0], dtype=float)
        x = x[:n]; y = y[:n]
        m = np.isfinite(x) & np.isfinite(y)
        return (x[m], y[m]) if m.any() else (np.array([0.0]), np.array([0.0]))

    def _safe_span(arr, pad=0.05):
        arr = _finite_arr(arr)
        lo, hi = float(np.nanmin(arr)), float(np.nanmax(arr))
        if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
            lo, hi = -1.0, 1.0
        span = hi - lo
        return lo - pad*span, hi + pad*span

    # ---------- data ----------
    L = getattr(block, "left_eye_data", None)
    R = getattr(block, "right_eye_data", None)
    if L is None or R is None:
        raise ValueError("Block must have left_eye_data and right_eye_data dataframes loaded.")
    for df_name, df in (("left_eye_data", L), ("right_eye_data", R)):
        for c in (cols.ms, cols.phi, cols.theta, cols.pupil):
            if c not in df.columns:
                raise ValueError(f"{df_name} missing required column '{c}'")

    # Left
    L_ms = _finite_arr(L[cols.ms]); L_phi = _finite_arr(L[cols.phi])
    L_theta = _finite_arr(L[cols.theta]); L_pupil = _finite_arr(L[cols.pupil])
    L_ms_phi, L_phi = _finite_xy(L_ms, L_phi)
    L_ms_theta, L_theta = _finite_xy(L_ms, L_theta)
    L_ms_pupil, L_pupil = _finite_xy(L_ms, L_pupil)
    srcL_phi   = ColumnDataSource(dict(ms=L_ms_phi,   val=L_phi))
    srcL_theta = ColumnDataSource(dict(ms=L_ms_theta, val=L_theta))
    srcL_pupil = ColumnDataSource(dict(ms=L_ms_pupil, val=L_pupil))

    # Right
    R_ms = _finite_arr(R[cols.ms]); R_phi = _finite_arr(R[cols.phi])
    R_theta = _finite_arr(R[cols.theta]); R_pupil = _finite_arr(R[cols.pupil])
    R_ms_phi, R_phi = _finite_xy(R_ms, R_phi)
    R_ms_theta, R_theta = _finite_xy(R_ms, R_theta)
    R_ms_pupil, R_pupil = _finite_xy(R_ms, R_pupil)
    srcR_phi   = ColumnDataSource(dict(ms=R_ms_phi,   val=R_phi))
    srcR_theta = ColumnDataSource(dict(ms=R_ms_theta, val=R_theta))
    srcR_pupil = ColumnDataSource(dict(ms=R_ms_pupil, val=R_pupil))

    # Navigator axis
    ms_all = pd.concat([pd.Series(L[cols.ms]), pd.Series(R[cols.ms])], ignore_index=True)
    ms_all = pd.to_numeric(ms_all, errors="coerce")
    ms_all = ms_all[np.isfinite(ms_all)]
    nav_x = np.sort(ms_all.values) if not ms_all.empty else np.array([0.0], dtype=float)
    nav_y = np.zeros_like(nav_x, dtype=float)
    nav_src = ColumnDataSource(dict(x=nav_x, y=nav_y))

    # ---------- figures ----------
    xlo, xhi = _safe_span(ms_all if not ms_all.empty else [0.0, 1.0])
    shared_x = Range1d(start=xlo, end=xhi)

    p_phi = figure(width=width, height=lane_height, x_range=shared_x,
                   tools="pan,wheel_zoom,box_zoom,reset,save", title="φ (phi): L & R")
    p_theta = figure(width=width, height=lane_height, x_range=shared_x,
                     tools="pan,wheel_zoom,box_zoom,reset,save", title="θ (theta): L & R")
    p_pupil = figure(width=width, height=lane_height, x_range=shared_x,
                     tools="pan,wheel_zoom,box_zoom,reset,save", title="Pupil diameter: L & R",
                     x_axis_label="time (ms)")

    for p in (p_phi, p_theta, p_pupil):
        p.add_tools(HoverTool(tooltips=[("t (ms)", "$x")], mode="vline"))

    p_phi.line("ms", "val", source=srcL_phi,   line_width=1.5, color="#d62728", legend_label="L φ")
    p_phi.line("ms", "val", source=srcR_phi,   line_width=1.5, color="#1f77b4", legend_label="R φ")
    p_theta.line("ms", "val", source=srcL_theta, line_width=1.5, color="#d62728", legend_label="L θ")
    p_theta.line("ms", "val", source=srcR_theta, line_width=1.5, color="#1f77b4", legend_label="R θ")
    p_pupil.line("ms", "val", source=srcL_pupil, line_width=1.5, color="#d62728", legend_label="L pupil")
    p_pupil.line("ms", "val", source=srcR_pupil, line_width=1.5, color="#1f77b4", legend_label="R pupil")

    for p in (p_phi, p_theta, p_pupil):
        p.legend.location = "top_left"

    ylo_phi, yhi_phi = _safe_span(pd.concat([pd.to_numeric(L[cols.phi], errors="coerce"),
                                             pd.to_numeric(R[cols.phi], errors="coerce")]))
    ylo_theta, yhi_theta = _safe_span(pd.concat([pd.to_numeric(L[cols.theta], errors="coerce"),
                                                 pd.to_numeric(R[cols.theta], errors="coerce")]))
    ylo_pupil, yhi_pupil = _safe_span(pd.concat([pd.to_numeric(L[cols.pupil], errors="coerce"),
                                                 pd.to_numeric(R[cols.pupil], errors="coerce")]))
    p_phi.y_range = Range1d(ylo_phi, yhi_phi)
    p_theta.y_range = Range1d(ylo_theta, yhi_theta)
    p_pupil.y_range = Range1d(ylo_pupil, yhi_pupil)

    # Navigator + RangeTool
    p_nav = figure(width=width, height=nav_height, y_range=(-1, 1),
                   tools="", toolbar_location=None, title="Navigator")
    p_nav.line("x", "y", source=nav_src, line_alpha=0.0)
    rt = RangeTool(x_range=shared_x)
    rt.overlay.fill_color = "green"; rt.overlay.fill_alpha = 0.15
    p_nav.add_tools(rt); p_nav.ygrid.grid_line_color = None

    # ---------- events (preload/empty) ----------
    if preload_df is not None and len(preload_df) > 0:
        df0 = preload_df.copy() if isinstance(preload_df, pd.DataFrame) else pd.DataFrame(preload_df)
        for k in ["animal", "block", "eye", "start_ms", "end_ms"]:
            if k not in df0.columns:
                df0[k] = np.nan
        df0["animal"]   = str(block.animal_call)
        df0["block"]    = str(block.block_num)
        df0["eye"]      = df0["eye"].astype(str).str.upper().fillna(default_eye)
        df0["start_ms"] = pd.to_numeric(df0["start_ms"], errors="coerce").astype(float)
        df0["end_ms"]   = pd.to_numeric(df0["end_ms"],   errors="coerce").astype(float)
        df0["note"]     = (df0["note"].astype(str) if "note" in df0.columns else "")
        df0 = df0.dropna(subset=["start_ms", "end_ms"])
        df0 = df0[df0["end_ms"] > df0["start_ms"]].reset_index(drop=True)
    else:
        df0 = pd.DataFrame(columns=["animal", "block", "eye", "start_ms", "end_ms", "note"])

    events_src = ColumnDataSource(dict(
        animal=[str(block.animal_call)] * len(df0),
        block=[str(block.block_num)] * len(df0),
        eye=(df0["eye"].astype(str).str.upper().tolist() if len(df0) else []),
        start_ms=(df0["start_ms"].astype(float).tolist() if len(df0) else []),
        end_ms=(df0["end_ms"].astype(float).tolist() if len(df0) else []),
        note=(df0["note"].astype(str).tolist() if len(df0) else []),
    ))

    # Overlay sources
    src_evt_LR = ColumnDataSource(dict(left=[], right=[], bottom=[], top=[]))
    src_evt_L  = ColumnDataSource(dict(left=[], right=[], bottom=[], top=[]))
    src_evt_R  = ColumnDataSource(dict(left=[], right=[], bottom=[], top=[]))

    # Shaded quads (source first, then glyph)
    for p in (p_phi, p_theta, p_pupil):
        p.add_glyph(src_evt_LR, Quad(left="left", right="right", bottom="bottom", top="top",
                                     fill_color="purple", fill_alpha=0.12))
        p.add_glyph(src_evt_L,  Quad(left="left", right="right", bottom="bottom", top="top",
                                     fill_color="red", fill_alpha=0.18))
        p.add_glyph(src_evt_R,  Quad(left="left", right="right", bottom="bottom", top="top",
                                     fill_color="blue", fill_alpha=0.18))

    # Initial overlays
    def _pack(df, eye_code):
        if df is None or df.empty:
            return dict(left=[], right=[], bottom=[], top=[])
        sub = df.copy()
        sub["start_ms"] = pd.to_numeric(sub["start_ms"], errors="coerce").astype(float)
        sub["end_ms"]   = pd.to_numeric(sub["end_ms"],   errors="coerce").astype(float)
        sub["eye"]      = sub["eye"].astype(str).str.upper()
        sub = sub[(np.isfinite(sub["start_ms"])) & (np.isfinite(sub["end_ms"])) & (sub["end_ms"] > sub["start_ms"])]
        sub = sub[sub["eye"] == eye_code]
        return dict(left=sub["start_ms"].tolist(), right=sub["end_ms"].tolist(),
                    bottom=[0.0]*len(sub), top=[1.0]*len(sub))

    src_evt_LR.data = _pack(df0, "LR")
    src_evt_L.data  = _pack(df0, "L")
    src_evt_R.data  = _pack(df0, "R")

    # ---------- controls ----------
    add_btn = Button(label="Add (nav range → event)", button_type="success", width=260)
    del_last_btn = Button(label="Delete last", button_type="warning", width=130)
    clear_btn = Button(label="Clear all", button_type="danger", width=120)
    dl_btn = Button(label="Download CSV", button_type="primary", width=150)
    copy_btn = Button(label="Copy JSON", button_type="default", width=120)
    eye_sel = Select(title="Eye", value=default_eye.upper(), options=["LR", "L", "R"], width=80)
    note_in = TextInput(title="Note", value="", width=320)
    fname_in = TextInput(title="Filename (no extension)",
                         value=f"{block.animal_call}_B{block.block_num}_manual_events",
                         width=350)

    # Rebuild overlays when events change
    rebuild_code = """
    const ev = events_src.data;
    const N = ev.start_ms.length;
    const LR = {left:[], right:[], bottom:[], top:[]};
    const L  = {left:[], right:[], bottom:[], top:[]};
    const R  = {left:[], right:[], bottom:[], top:[]};
    for (let i=0; i<N; i++) {
      const s = Number(ev.start_ms[i]);
      const e = Number(ev.end_ms[i]);
      const eye = String(ev.eye[i] || "").toUpperCase();
      if (!isFinite(s) || !isFinite(e) || e <= s) continue;
      const tgt = eye === "LR" ? LR : (eye === "L" ? L : (eye === "R" ? R : null));
      if (tgt) { tgt.left.push(s); tgt.right.push(e); tgt.bottom.push(0.0); tgt.top.push(1.0); }
    }
    src_evt_LR.data = LR; src_evt_L.data = L; src_evt_R.data = R;
    src_evt_LR.change.emit(); src_evt_L.change.emit(); src_evt_R.change.emit();
    """
    rebuild_js = CustomJS(args=dict(events_src=events_src,
                                    src_evt_LR=src_evt_LR, src_evt_L=src_evt_L, src_evt_R=src_evt_R),
                          code=rebuild_code)
    events_src.js_on_change("data", rebuild_js)

    # Add event from current shared range
    add_code = """
    const s = shared_x.start, e = shared_x.end;
    if (!(isFinite(s) && isFinite(e) && e > s)) return;
    const eye = String(eye_sel.value || "LR").toUpperCase();
    const note = String(note_in.value || "");
    const ev = events_src.data;
    ev.animal.push(animal_id);
    ev.block.push(block_id);
    ev.eye.push(eye);
    ev.start_ms.push(s);
    ev.end_ms.push(e);
    ev.note.push(note);
    events_src.change.emit();
    """
    add_btn.js_on_click(CustomJS(args=dict(events_src=events_src, shared_x=shared_x, eye_sel=eye_sel,
                                           note_in=note_in,
                                           animal_id=str(block.animal_call), block_id=str(block.block_num)),
                                 code=add_code))

    # Delete last
    del_last_code = """
    const ev = events_src.data;
    const N = ev.start_ms.length;
    if (N <= 0) return;
    for (const k in ev) { if (ev.hasOwnProperty(k)) ev[k].splice(N-1, 1); }
    events_src.change.emit();
    """
    del_last_btn.js_on_click(CustomJS(args=dict(events_src=events_src), code=del_last_code))

    # Clear all
    clear_code = """
    const ev = events_src.data;
    for (const k in ev) { if (ev.hasOwnProperty(k)) ev[k] = []; }
    events_src.change.emit();
    """
    clear_btn.js_on_click(CustomJS(args=dict(events_src=events_src), code=clear_code))

    # Download CSV to browser
    dl_code = """
    const ev = events_src.data;
    const cols = ["animal","block","eye","start_ms","end_ms","note"];
    const N = ev.start_ms.length;
    const rows = [cols.join(",")];
    for (let i=0; i<N; i++) {
      const out = [
        String(ev.animal[i] ?? ""),
        String(ev.block[i] ?? ""),
        String(ev.eye[i] ?? ""),
        String(ev.start_ms[i] ?? ""),
        String(ev.end_ms[i] ?? ""),
        (String(ev.note[i] ?? "").replace(/"/g,'""'))
      ];
      out[5] = `"${out[5]}"`;
      rows.push(out.join(","));
    }
    const csv = rows.join("\\n");
    const blob = new Blob([csv], {type: "text/csv;charset=utf-8;"});
    const url = URL.createObjectURL(blob);
    const a = document.createElement("a");
    const base = (fname_in.value || "manual_events").trim();
    a.href = url; a.download = `${base}.csv`; a.style.display = "none";
    document.body.appendChild(a); a.click(); document.body.removeChild(a);
    URL.revokeObjectURL(url);
    """
    dl_btn.js_on_click(CustomJS(args=dict(events_src=events_src, fname_in=fname_in), code=dl_code))

    # Copy JSON to clipboard (quick paste back into a cell if you want)
    copy_code = """
    const ev = events_src.data;
    const N = ev.start_ms.length;
    const rows = [];
    for (let i=0; i<N; i++) {
      rows.push({
        animal: String(ev.animal[i] ?? ""),
        block: String(ev.block[i] ?? ""),
        eye: String(ev.eye[i] ?? ""),
        start_ms: Number(ev.start_ms[i]),
        end_ms: Number(ev.end_ms[i]),
        note: String(ev.note[i] ?? "")
      });
    }
    const txt = JSON.stringify(rows, null, 2);
    navigator.clipboard.writeText(txt).then(()=>{},()=>{});
    """
    copy_btn.js_on_click(CustomJS(args=dict(events_src=events_src), code=copy_code))

    # ---------- live table ----------
    cols_def = [
        TableColumn(field="animal",   title="animal",   formatter=StringFormatter(text_align="left")),
        TableColumn(field="block",    title="block",    formatter=StringFormatter(text_align="left")),
        TableColumn(field="eye",      title="eye",      formatter=StringFormatter(text_align="center")),
        TableColumn(field="start_ms", title="start_ms", formatter=NumberFormatter(format="0.000")),
        TableColumn(field="end_ms",   title="end_ms",   formatter=NumberFormatter(format="0.000")),
        TableColumn(field="note",     title="note",     formatter=StringFormatter(text_align="left")),
    ]
    table = DataTable(source=events_src, columns=cols_def, width=width, height=220, index_position=None,
                      sizing_mode="stretch_width", sortable=True, selectable=True, editable=False)

    # ---------- layout ----------
    top_note = Div(text=f"<b>{block.animal_call} B{block.block_num}</b> — Manual Tagger (JS-only)")
    controls = row(add_btn, del_last_btn, clear_btn, dl_btn, copy_btn, eye_sel, note_in, fname_in)
    lanes = gridplot([[p_phi], [p_theta], [p_pupil]], toolbar_location="above", merge_tools=False)
    layout = column(top_note, lanes, p_nav, controls, Div(text="<b>Annotations</b>"), table,
                    sizing_mode="stretch_width")

    # Trigger initial overlay build
    events_src.data = dict(events_src.data)

    show(layout)

    return dict(
        figures=(p_phi, p_theta, p_pupil, p_nav),
        events_source=events_src,
        overlay_sources=(src_evt_LR, src_evt_L, src_evt_R),
        shared_x=shared_x,
        table=table,
    )


In [ ]:
# manual tagging function - zoom in to the suspect samples and tag them, then copy the json output and paste in the next cell
ui = interactive_manual_tagger_js(block, preload_df=None, default_eye="LR")

In [ ]:
import json

json_text = """[
  {
    "animal": "PV_106",
    "block": "015",
    "eye": "LR",
    "start_ms": 95601.87183348203,
    "end_ms": 112831.52235069776,
    "note": ""
  },
  {
    "animal": "PV_106",
    "block": "015",
    "eye": "LR",
    "start_ms": 150395.8051821393,
    "end_ms": 167625.45569935502,
    "note": ""
  }
]"""
events_df = pd.DataFrame(json.loads(json_text))

## Step 7: Integrate Manual Events into Block Annotations

This cell integrates manually created events into the per-block annotation CSV file. The `integrate_manual_events_into_block_annotations()` function:
- Normalizes the events dataframe
- Merges with existing annotations
- Writes to the block's `manual_event_annotations.csv` file

In [ ]:
# now visually tag the manually annotated samples:

reviewed_df = review_events_multi_with_arena_v2(block_dict, events_df)

In [ ]:
# === Integrate manual events into per-block annotations CSV (uses your existing IO helpers) ===
from __future__ import annotations
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional, Tuple

def integrate_manual_events_into_block_annotations(
    block,
    events_df: pd.DataFrame,
    *,
    default_tag: Optional[bool] = None,   # if provided, assign TAG_COL to all new rows; else leave as None
    overwrite: bool = True,               # pass-through to your _merge_annotations logic
    round_ms: float = 1.0,                # used only for light validation; merge uses your internal rounding
) -> Tuple[pd.DataFrame, Path]:
    """
    Normalize a manual events_df (['animal','block','eye','start_ms','end_ms']) and merge it into this block's
    manual_event_annotations.csv under block.analysis_path using your existing helpers.

    Behavior:
    - Fills missing animal/block from the provided block.
    - Coerces types, drops invalid rows (NaN or end<=start).
    - Sets TAG_COL to `default_tag` (if provided) and TS_COL to a fresh timestamp only when a tag is set now.
    - Delegates merge+write to _write_block_annotations(..., overwrite=<flag>), which calls _merge_annotations.

    Returns
    -------
    merged_df : pd.DataFrame
        The full, merged on-disk table after the write.
    out_path : Path
        Path to the updated CSV (block.analysis_path / CSV_NAME).
    """
    # --- sanity and paths ---
    if not hasattr(block, "analysis_path"):
        raise AttributeError("block.analysis_path is required and must be a pathlib.Path")
    analysis_path: Path = Path(block.analysis_path)
    analysis_path.mkdir(parents=True, exist_ok=True)
    out_path = analysis_path / CSV_NAME  # uses your global CSV_NAME

    # --- normalize incoming events ---
    need = ["animal","block","eye","start_ms","end_ms"]
    for c in need:
        if c not in events_df.columns:
            events_df[c] = np.nan

    df = events_df[need].copy()

    # fill animal/block from block if missing/empty
    blk_animal = str(getattr(block, "animal_call", ""))
    blk_block  = str(getattr(block, "block_num", ""))
    if df["animal"].isna().all() or (df["animal"].astype(str).str.len()==0).all():
        df["animal"] = blk_animal
    if df["block"].isna().all() or (df["block"].astype(str).str.len()==0).all():
        df["block"] = blk_block

    # dtypes
    df["animal"] = df["animal"].astype(str)
    df["block"]  = df["block"].astype(str)
    df["eye"]    = df["eye"].astype(str).str.upper().str.strip()  # keep 'L','R','LR' as-is
    df["start_ms"] = pd.to_numeric(df["start_ms"], errors="coerce").astype(float)
    df["end_ms"]   = pd.to_numeric(df["end_ms"], errors="coerce").astype(float)

    # drop invalid
    df = df.dropna(subset=["start_ms","end_ms"]).copy()
    df = df[df["end_ms"] > df["start_ms"]].copy()

    # optional light sanity: warn on absurdly small/negative rounded spans
    if not df.empty and round_ms is not None and round_ms > 0:
        # nothing to do beyond basic validation; your merge function handles rounding internally
        pass

    # --- build final schema expected by your per-block CSV ---
    out_block = pd.DataFrame({
        "animal_call": df["animal"].astype(str),
        "block":       df["block"].astype(str),
        "eye":         df["eye"].astype(str),
        "start_ms":    df["start_ms"].astype(float),
        "end_ms":      df["end_ms"].astype(float),
    })

    # tag + timestamp columns
    def _now_stamp() -> str:
        return pd.Timestamp.now().strftime("%Y_%m_%d_%H_%M")

    if default_tag is None:
        out_block[TAG_COL] = pd.Series([None]*len(out_block), dtype="object")
        out_block[TS_COL]  = pd.Series([None]*len(out_block), dtype="object")
    else:
        out_block[TAG_COL] = bool(default_tag)
        out_block[TS_COL]  = _now_stamp()

    # --- delegate merge+write using your helper (merges on rounded start/end, overwriting if requested) ---
    out_path = _write_block_annotations(block, out_block, overwrite=overwrite)

    # load the final on-disk merged table to return (consistent with your reader)
    merged_df = load_block_annotations(block)
    return merged_df, out_path


# ===== Example usage =====
merged, path = integrate_manual_events_into_block_annotations(
    block,
    events_df,          # from manual tagger (columns: animal, block, eye, start_ms, end_ms)
    default_tag=None,   # or True/False if you want to initialize all as BAD/GOOD immediately
    overwrite=True,     # if an event matches start/end it will be replaced by the new row
)
print("Updated CSV:", path)
display(merged.head())


### 8.2: Export Clean Eye Data

This cell exports the cleaned eye data to CSV files using `export_clean_eye_data()`. 

**Options:**
- `overwrite_original=False`: Only exports `left_eye_data_clean.csv` and `right_eye_data_clean.csv`
- `overwrite_original=True`: Also overwrites `block.left_eye_data` and `block.right_eye_data` with the cleaned versions

**Output files:**
- `left_eye_data_clean.csv` - Cleaned left eye data
- `right_eye_data_clean.csv` - Cleaned right eye data

In [ ]:
# THIS IS WHERE WE MAKE CLEAN EYE_DFs
import numpy as np
import pandas as pd
from typing import Optional, Tuple, Dict

# assumes EyeColumns dataclass, TAG_COL, load_block_annotations() exist in scope

def apply_manual_outlier_cleanup(
    block,
    annotations_df: Optional[pd.DataFrame] = None,
    *,
    cols: EyeColumns = EyeColumns(),
    bad_col: str = TAG_COL,            # "manual_outlier_detected"
    value_cols: Tuple[str, str, str] = ("k_theta", "k_phi", "pupil_diameter"),
    add_mask_columns: bool = True,     # add boolean mask columns to the *clean* dfs for audit
) -> Dict[str, pd.DataFrame]:
    """
    Create cleaned eye-data copies on the block by setting 'bad' intervals to NaN in value_cols.
    'Bad' is defined by the GUI review's per-interval tag bad_col == True.

    Inputs
    ------
    block : BlockSync-like object with:
        - animal_call, block_num
        - left_eye_data, right_eye_data (DataFrames with cols.ms present)
    annotations_df : optional DataFrame of events with columns:
        ['animal_call','block','eye','start_ms','end_ms', bad_col]
        If None, loads from this block's per-block CSV via load_block_annotations(block).
    cols : EyeColumns
        Names for ms, phi, theta, pupil in the eye DataFrames (defaults match your pipeline).
    bad_col : str
        Column name in annotations_df whose True values denote BAD intervals to wipe.
    value_cols : tuple
        The columns that will be set to NaN inside BAD intervals.
    add_mask_columns : bool
        If True, adds a boolean 'clean_badmask' column to each clean df for traceability.

    Side effects
    -----------
    Sets on `block`:
        - block.left_eye_data_clean  (pd.DataFrame)
        - block.right_eye_data_clean (pd.DataFrame)
      If block has `left_eye_df`/`right_eye_df`, also sets:
        - block.left_eye_df_clean / block.right_eye_df_clean

    Returns
    -------
    dict with keys: {'left_eye_data_clean','right_eye_data_clean'}
    """
    # --- resolve annotations ---
    if annotations_df is None:
        annotations_df = load_block_annotations(block)

    if annotations_df is None or annotations_df.empty:
        # nothing to wipe; just copy originals
        L_clean = getattr(block, "left_eye_data").copy()
        R_clean = getattr(block, "right_eye_data").copy()
        if add_mask_columns:
            L_clean["clean_badmask"] = False
            R_clean["clean_badmask"] = False
        setattr(block, "left_eye_data_clean", L_clean)
        setattr(block, "right_eye_data_clean", R_clean)
        # optional compatibility mirror
        if hasattr(block, "left_eye_df"):
            setattr(block, "left_eye_df_clean", L_clean.copy())
        if hasattr(block, "right_eye_df"):
            setattr(block, "right_eye_df_clean", R_clean.copy())
        return {"left_eye_data_clean": L_clean, "right_eye_data_clean": R_clean}

    # --- filter to this block only ---
    a_str = str(block.animal_call)
    b_str = str(block.block_num)
    ann = annotations_df.copy()

    # normalize expected columns
    needed = {"animal_call","block","eye","start_ms","end_ms", bad_col}
    missing = [c for c in needed if c not in ann.columns]
    if missing:
        raise ValueError(f"annotations_df missing required columns: {missing}")

    ann["animal_call"] = ann["animal_call"].astype(str)
    ann["block"] = ann["block"].astype(str)
    ann = ann[(ann["animal_call"] == a_str) & (ann["block"] == b_str)]

    if ann.empty:
        # no intervals for this block
        L_clean = getattr(block, "left_eye_data").copy()
        R_clean = getattr(block, "right_eye_data").copy()
        if add_mask_columns:
            L_clean["clean_badmask"] = False
            R_clean["clean_badmask"] = False
        setattr(block, "left_eye_data_clean", L_clean)
        setattr(block, "right_eye_data_clean", R_clean)
        if hasattr(block, "left_eye_df"):
            setattr(block, "left_eye_df_clean", L_clean.copy())
        if hasattr(block, "right_eye_df"):
            setattr(block, "right_eye_df_clean", R_clean.copy())
        return {"left_eye_data_clean": L_clean, "right_eye_data_clean": R_clean}

    # Keep only rows explicitly tagged BAD == True
    def _coerce_bool(v):
        if isinstance(v, bool): return v
        s = str(v).strip().lower()
        if s in {"true","t","1","yes","y"}: return True
        if s in {"false","f","0","no","n","nan","none",""}: return False
        return False

    ann["_is_bad"] = ann[bad_col].map(_coerce_bool)
    ann_bad = ann[ann["_is_bad"]].copy()
    if ann_bad.empty:
        # nothing to mask
        L_clean = getattr(block, "left_eye_data").copy()
        R_clean = getattr(block, "right_eye_data").copy()
        if add_mask_columns:
            L_clean["clean_badmask"] = False
            R_clean["clean_badmask"] = False
        setattr(block, "left_eye_data_clean", L_clean)
        setattr(block, "right_eye_data_clean", R_clean)
        if hasattr(block, "left_eye_df"):
            setattr(block, "left_eye_df_clean", L_clean.copy())
        if hasattr(block, "right_eye_df"):
            setattr(block, "right_eye_df_clean", R_clean.copy())
        return {"left_eye_data_clean": L_clean, "right_eye_data_clean": R_clean}

    # --- pull eye data, sanity checks ---
    L = getattr(block, "left_eye_data")
    R = getattr(block, "right_eye_data")
    for df_name, df in (("left_eye_data", L), ("right_eye_data", R)):
        if cols.ms not in df.columns:
            raise ValueError(f"{df_name} missing time column '{cols.ms}'")
        for vc in value_cols:
            if vc not in df.columns:
                raise ValueError(f"{df_name} missing value column '{vc}'")

    # --- build masks from intervals ---
    msL = pd.to_numeric(L[cols.ms], errors="coerce").values.astype(float)
    msR = pd.to_numeric(R[cols.ms], errors="coerce").values.astype(float)
    maskL = np.zeros(msL.shape, dtype=bool)
    maskR = np.zeros(msR.shape, dtype=bool)

    # Accept eye in {'L','R','LR'} (case-insensitive); treat 'LR' as both eyes
    for _, row in ann_bad.iterrows():
        s = float(row["start_ms"])
        e = float(row["end_ms"])
        eye = str(row["eye"]).upper().strip() if pd.notna(row["eye"]) else "LR"
        if e < s:
            s, e = e, s  # swap if misordered

        if eye in ("L", "LR"):
            maskL |= (msL >= s) & (msL <= e)
        if eye in ("R", "LR"):
            maskR |= (msR >= s) & (msR <= e)

    # --- create cleaned copies and set NaNs in requested columns ---
    L_clean = L.copy()
    R_clean = R.copy()
    for vc in value_cols:
        L_clean.loc[maskL, vc] = np.nan
        R_clean.loc[maskR, vc] = np.nan

    if add_mask_columns:
        L_clean["clean_badmask"] = maskL
        R_clean["clean_badmask"] = maskR

    # --- attach back to block (primary names) ---
    setattr(block, "left_eye_data_clean", L_clean)
    setattr(block, "right_eye_data_clean", R_clean)

    # --- optional compatibility: if the project also uses *_eye_df names, mirror the cleans ---
    if hasattr(block, "left_eye_df"):
        setattr(block, "left_eye_df_clean", L_clean.copy())
    if hasattr(block, "right_eye_df"):
        setattr(block, "right_eye_df_clean", R_clean.copy())

    return {"left_eye_data_clean": L_clean, "right_eye_data_clean": R_clean}


In [ ]:
# Export the cleaned data to CSV files
# Set overwrite_original=True if you want to also overwrite block.left_eye_data and block.right_eye_data
export_clean_eye_data(block, overwrite_original=False)

In [ ]:
apply_manual_outlier_cleanup(block)